In [20]:
# 1) load everything
import pandas as pd

sheets = pd.read_excel("../data/online_retail_II.xlsx", sheet_name=None) # reads both sheets, ../data -> go up one folder
df = pd.concat(sheets.values(), ignore_index=True) # stacks both sheets into one table renumbering from 0

df.columns = ["invoice", "stock_code", "description", "quantity", "invoice_date", "price", "customer_id", "country"] # renames columns to avoid awkwardness in SQL
df.to_pickle("../data/raw.pkl") # saves fast-loading copy
df.shape

(1067371, 8)

In [21]:
# 2a) first look
df.head(10) # shows first 10 rows

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom


In [22]:
# 2b) first look 
df.info() # lists each column/data type/number of values

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   invoice       1067371 non-null  object        
 1   stock_code    1067371 non-null  object        
 2   description   1062989 non-null  object        
 3   quantity      1067371 non-null  int64         
 4   invoice_date  1067371 non-null  datetime64[us]
 5   price         1067371 non-null  float64       
 6   customer_id   824364 non-null   float64       
 7   country       1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB


In [23]:
# 3) data quality table
# each line asks yes/no for each row and counts the yeses
checks = { 
    "Total rows": len(df),
    "Exact duplicate rows": df.duplicated().sum(),
    "Missing customer ID": df["customer_id"].isna().sum(), # isna flags empty cells
    "Missing description": df["description"].isna().sum(),
    "Cancellation lines (invoice starts with C)": df["invoice"].astype(str).str.startswith("C").sum(), # turns invoice num into text
    "Zero or negative quantity": (df["quantity"] <= 0).sum(),
    "Zero or negative price": (df["price"] <= 0).sum(),
}

report = pd.Series(checks, name="rows").to_frame() # turns into a table
report["pct_of_total"] = (report["rows"] / len(df) * 100).round(2) # adds percentage column
report.to_csv("../reports/data_quality_summary.csv", index_label="check") # saves table to reports folder
report

,rows,pct_of_total
Total rows,1067371,100.00
Exact duplicate rows,34335,3.22
Missing customer ID,243007,22.77
Missing description,4382,0.41
Cancellation lines (invoice starts with C),19494,1.83
Zero or negative quantity,22950,2.15
Zero or negative price,6207,0.58


In [24]:
# 4) scope of the data
print("Date range:", df["invoice_date"].min(), "to", df["invoice_date"].max())
print("Customers:", df["customer_id"].nunique()) # nunique() counts distinct values
print("Countries:", df["country"].nunique())
print("Products:", df["stock_code"].nunique())

Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Customers: 5942
Countries: 43
Products: 5305


In [ ]:
# 5) tag every row with its issues
original_columns = ["invoice", "stock_code", "description", "quantity", "invoice_date", "price", "customer_id", "country"] 

df ["stock_code"] = df["stock_code"].astype(str).str.strip() # turned all product codes into text for simplicity without space

# runs T/F test on each row and stores answers in new column
df["is_duplicate"] = df.duplicated(subset=original_columns) # subset tells duplicate checks to compare only 8 og columns from excel file to avoid wrongly marking duplicates
df["is_cancellation"] = df["invoice"].astype(str).str.startswith("C")
df["missing_customer"] = df["customer_id"].isna()
df["missing_description"] = df["description"].isna()
df["bad_quantity"] = df["quantity"] <= 0
df["bad_price"] = df["price"] <= 0

standard_code = df["stock_code"].str.match(r"^\d{5}[A-Za-z]*$")
other_product = df["stock_code"].str.startswith("DCGS") | (df["stock_code"] == "PADS") # refined after seeing DCGS and PADS in step 8
df["non_product_code"] = ~(standard_code | other_product)

In [26]:
# 6) count rows with > 0 issue

flags = ["is_duplicate", "is_cancellation", "missing_customer", "missing_description", "bad_quantity", "bad_price", "non_product_code"]

df["issue_count"] = df[flags].sum(axis=1) # adds across each row
has_issue = df["issue_count"] > 0

print("Rows with at least one issue:", has_issue.sum())
print("Percentage of all rows with at least one issue:", round(has_issue.mean() * 100, 2), "%") # same number as percentage
df["issue_count"].value_counts().sort_index() # shows how many rows have 0, 1, 2... issues. 

Rows with at least one issue: 290794
Percentage of all rows with at least one issue: 27.24 %


issue_count
0    776577
1    255723
2     27906
3      3955
4      3139
5        71
Name: count, dtype: int64

In [27]:
# 7) Negative quantities that aren't cancellations
odd = df[df["bad_quantity"] & ~df["is_cancellation"]] # filters table 
print("Rows:", len(odd))
print("With a zero price:", (odd["price"] == 0).sum())
print("With no customer id:", (odd["missing_customer"].sum()))
odd["description"].value_counts(dropna=False).head(15) # lists 15 most common descriptions including blanks

Rows: 3457
With a zero price: 3457
With no customer id: 3457


description
NaN                       2689
check                      123
damages                     84
?                           83
damaged                     78
missing                     27
sold as set on dotcom       20
Damaged                     17
smashed                      9
thrown away                  9
Unsaleable, destroyed.       9
dotcom                       8
damages?                     7
??                           7
crushed                      6
Name: count, dtype: int64

In [28]:
# 8) Non-product codes
df.loc[df["non_product_code"], ["stock_code", "description"]].value_counts(dropna=False).head(20)

stock_code    description                        
POST          POSTAGE                                2115
DOT           DOTCOM POSTAGE                         1444
M             Manual                                 1421
C2            CARRIAGE                                279
D             Discount                                177
S             SAMPLES                                 104
BANK CHARGES  Bank Charges                             96
AMAZONFEE     AMAZON FEE                               43
ADJUST        Adjustment by john on 26/01/2010 16      38
gift_0001_20  Dotcomgiftshop Gift Voucher £20.00       26
ADJUST        Adjustment by john on 26/01/2010 17      26
gift_0001_30  Dotcomgiftshop Gift Voucher £30.00       24
PADS          PADS TO MATCH ALL CUSHIONS               19
CRUK          CRUK Commission                          16
TEST001       This is a test product.                  15
gift_0001_10  Dotcomgiftshop Gift Voucher £10.00       14
POST          NaN     

In [29]:
# 9) How much revenue has no customer attached
df["line_total"] = df["quantity"] * df["price"] # total revenue per line
sales = df[~df["is_duplicate"] & ~df["is_cancellation"] & ~df["bad_quantity"] & ~df["bad_price"] & ~df["non_product_code"]] # remove all flags to keep all rows that look like real sales

by_customer = sales.groupby("missing_customer")["line_total"].agg(rows="count", revenue="sum") # calculates count of T and F and also sum of each group
by_customer.index = ["Has customer ID", "No customer ID"] # replaces row names
by_customer["pct_of_revenue"] = (by_customer["revenue"] / by_customer["revenue"].sum() * 100).round(2)
by_customer

,rows,revenue,pct_of_revenue
Has customer ID,776577,17068567.97,86.89
No customer ID,226760,2575262.77,13.11


In [30]:
# 10) saves table in data folder
df.to_pickle("../data/flagged.pkl")

In [31]:
# run all non-product code
codes = df.loc[df["non_product_code"], "stock_code"].value_counts()
print(len(codes), "distinct codes")
print(codes.to_string())

29 distinct codes
stock_code
POST            2122
DOT             1446
M               1421
C2               282
D                177
S                104
BANK CHARGES     102
ADJUST            67
AMAZONFEE         43
gift_0001_20      29
gift_0001_30      29
PADS              19
gift_0001_10      16
CRUK              16
TEST001           15
gift_0001_50       8
gift_0001_40       7
B                  6
m                  5
gift_0001_80       4
gift_0001_70       3
ADJUST2            3
SP1002             3
TEST002            2
gift_0001_60       2
gift_0001_90       2
GIFT               1
47503J             1
C3                 1
